In [2]:
import requests

url = "https://civicdb.org/api/graphql"
headers = {
    "Content-Type": "application/json",
}

query = """
query browseMolecularProfiles($after: String) {
  molecularProfiles(first: 300, after: $after) {
    edges {
      node {
        id
        name
        description
        molecularProfileScore
        variants {
          id
          name
        }
        assertions {
          nodes{
            id
            name
            description
            disease{
              id
              name
            } 
          }
        } 
      }
    }
    pageInfo {
      endCursor
      hasNextPage
    }
    totalCount
  }
}
"""

all_molecular_profiles = []
variables = {"after": None}

while True:
    response = requests.post(url, json={'query': query, 'variables': variables}, headers=headers)
    response_json = response.json()
    
    if 'data' in response_json:
        molecular_profiles = response_json["data"]["molecularProfiles"]["edges"]
        all_molecular_profiles.extend(molecular_profiles)
        
        page_info = response_json["data"]["molecularProfiles"]["pageInfo"]
        if not page_info["hasNextPage"]:
            break
        variables["after"] = page_info["endCursor"]
    else:
        print("Error in response:", response_json.get('errors'))
        break

print(f"Total profiles fetched: {len(all_molecular_profiles)}")

Total profiles fetched: 5038


In [3]:
molecular_profiles_filtered = [edge for edge in all_molecular_profiles if edge["node"]["molecularProfileScore"] != 0]

In [4]:
molecular_profiles_filtered

[{'node': {'id': 12,
   'name': 'BRAF V600E',
   'description': 'BRAF V600E has been shown to be recurrent in many cancer types. It is one of the most widely studied variants in cancer. This variant is correlated with poor prognosis in certain cancer types, including colorectal cancer and papillary thyroid cancer. The targeted therapeutic dabrafenib has been shown to be effective in clinical trials with an array of BRAF mutations and cancer types. Dabrafenib has also shown to be effective when combined with the MEK inhibitor trametinib in colorectal cancer and melanoma. However, in patients with TP53, CDKN2A and KRAS mutations, dabrafenib resistance has been reported. Ipilimumab, regorafenib, vemurafenib, and a number of combination therapies have been successful in treating V600E mutations. However, cetuximab and panitumumab have been largely shown to be ineffective without supplementary treatment.',
   'molecularProfileScore': 1433.5,
   'variants': [{'id': 12, 'name': 'V600E'}],
   

In [ ]:
import sqlite3

# Connect to the SQLite database (or create it if it doesn't exist)
conn = sqlite3.connect('database.db')
cursor = conn.cursor()

# Create a table to store the molecular profiles
cursor.execute('''
CREATE TABLE IF NOT EXISTS molecular_profiles (
    id TEXT PRIMARY KEY,
    name TEXT,
    description TEXT,
    molecularProfileScore REAL
)
''')

# Insert the filtered molecular profiles into the table
for profile in molecular_profiles_filtered:
    node = profile['node']
    cursor.execute('''
    INSERT OR REPLACE INTO molecular_profiles (id, name, description, molecularProfileScore)
    VALUES (?, ?, ?, ?)
    ''', (node['id'], node['name'], node['description'], node['molecularProfileScore']))

# Commit the transaction and close the connection
conn.commit()
conn.close()